# Volatility Surfaces: Building, Visualizing, and Calibrating

## Introduction

This notebook explores the fascinating world of implied volatility surfaces:

1. **Building** vol surfaces from market quotes
2. **Visualizing** the vol smile and term structure
3. **Understanding** arbitrage constraints
4. **Calibrating** parametric models (SABR)

---

### What is an Implied Volatility Surface?

When we invert the Black-Scholes formula using market option prices, we get **implied volatility**. If Black-Scholes were correct, all options on the same underlying would have the same implied vol. They don't!

**The vol surface** maps the variation of implied vol across:
- **Strike** (moneyness): Creates the "smile" or "skew"
- **Expiry** (term): Creates the "term structure"

---

### Why Does the Smile Exist?

| Cause | Effect on Smile |
|-------|----------------|
| **Fat tails** | Higher vol for OTM options |
| **Leverage effect** | Negative correlation spot-vol → skew |
| **Jump risk** | Crash premium for OTM puts |
| **Supply/demand** | Hedging pressure from structured products |

In [ ]:
# =============================================================================
# SETUP: Imports
# =============================================================================

import sys
from pathlib import Path
import numpy as np
from datetime import date

sys.path.insert(0, str(Path.cwd().parents[1]))

# Plotting
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# QuantStrata imports
from src.marketdata.surfaces.vol_surface import GridVolSurface, FlatVolSurface
from src.marketdata.surfaces.quotes import VolQuote

print("✓ All imports successful!")

## 1. Creating Vol Quotes from Market Data

In practice, vol quotes come from:
- **Options exchanges** (listed options)
- **OTC brokers** (FX, rates)
- **Composite feeds** (Bloomberg, Reuters)

FX markets typically quote in **delta convention**:
- 25-delta put (25D P)
- ATM (50D)
- 25-delta call (25D C)

We'll create a realistic FX vol surface for EURUSD.

In [ ]:
# =============================================================================
# STEP 1: Define Vol Quotes (EURUSD FX Surface)
# =============================================================================

# Market parameters
SPOT = 1.0850
R_DOMESTIC = 0.05  # USD
R_FOREIGN = 0.04   # EUR

# Vol quotes: Expiry x Delta grid
# Note: We'll use a smile with negative skew (typical for FX, equity)
# meaning OTM puts have higher vol than OTM calls

# Expiries in years
expiries = [1/52, 2/52, 1/12, 2/12, 3/12, 6/12, 1.0]  # 1W to 1Y
expiry_labels = ['1W', '2W', '1M', '2M', '3M', '6M', '1Y']

# Deltas (from put side to call side)
deltas = [0.10, 0.25, 0.50, 0.75, 0.90]  # 10D P to 90D (10D C)
delta_labels = ['10D Put', '25D Put', 'ATM', '25D Call', '10D Call']

# Vol matrix (expiry x delta)
# Building realistic smile: higher vols for wings, increasing with expiry
vol_matrix = np.array([
    # 10D P,  25D P,   ATM,   25D C,  10D C  <- Delta
    [0.125,  0.095,  0.082,  0.088,  0.115],  # 1W - short-dated, smile
    [0.122,  0.094,  0.083,  0.090,  0.118],  # 2W
    [0.120,  0.095,  0.085,  0.092,  0.116],  # 1M
    [0.118,  0.096,  0.088,  0.094,  0.114],  # 2M
    [0.116,  0.098,  0.090,  0.096,  0.112],  # 3M
    [0.115,  0.100,  0.094,  0.098,  0.111],  # 6M - term structure
    [0.118,  0.104,  0.098,  0.102,  0.114],  # 1Y
])

# Create VolQuote objects
vol_quotes = []
for i, exp in enumerate(expiries):
    for j, delta in enumerate(deltas):
        vol_quotes.append(VolQuote(
            expiry=exp,
            delta=delta,
            vol=vol_matrix[i, j],
        ))

print("Vol Quote Summary:")
print("="*70)
print(f"Number of quotes: {len(vol_quotes)}")
print(f"Expiry range: {expiry_labels[0]} to {expiry_labels[-1]}")
print(f"Delta range: {delta_labels[0]} to {delta_labels[-1]}")
print(f"\nVol matrix (rows=expiries, cols=deltas):")
print(f"{'Expiry':<8}", end='')
for d in delta_labels:
    print(f"{d:>10}", end='')
print()
print("-"*58)
for i, exp in enumerate(expiry_labels):
    print(f"{exp:<8}", end='')
    for j in range(len(deltas)):
        print(f"{vol_matrix[i,j]:>10.2%}", end='')
    print()

## 2. Visualizing the Vol Smile

The **volatility smile** shows how implied vol varies with strike at a fixed expiry.

Key features to observe:
- **Symmetric smile**: Both wings elevated (FX markets)
- **Skew**: Left wing higher than right (equity markets)
- **ATM level**: Center of the smile

In [ ]:
# =============================================================================
# STEP 2: Visualize the Vol Smile at Different Expiries
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.viridis(np.linspace(0, 0.9, len(expiries)))

for i, (exp_label, exp) in enumerate(zip(expiry_labels, expiries)):
    vols = vol_matrix[i, :]
    ax.plot(deltas, vols * 100, 'o-', color=colors[i], linewidth=2, 
            markersize=8, label=exp_label)

ax.set_xlabel('Delta', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title('EURUSD Volatility Smile by Expiry', fontsize=14, fontweight='bold')
ax.legend(title='Expiry', loc='upper center', ncol=4)
ax.set_xticks(deltas)
ax.set_xticklabels(['10Δ\n(OTM Put)', '25Δ\n(Put)', '50Δ\n(ATM)', '75Δ\n(Call)', '90Δ\n(OTM Call)'])
ax.grid(True, alpha=0.3)

# Add annotation
ax.annotate('Wings elevated\n(smile)', xy=(0.10, vol_matrix[0, 0] * 100), 
            xytext=(0.15, 13), fontsize=10, color='darkblue',
            arrowprops=dict(arrowstyle='->', color='darkblue', lw=1.5))

plt.tight_layout()
plt.show()

# Smile metrics
print("\nSmile Metrics (1M expiry):")
print(f"  ATM vol:           {vol_matrix[2, 2]:.2%}")
print(f"  25D Risk Reversal: {vol_matrix[2, 3] - vol_matrix[2, 1]:.2%} (25C - 25P)")
print(f"  25D Butterfly:     {(vol_matrix[2, 1] + vol_matrix[2, 3])/2 - vol_matrix[2, 2]:.2%} ((25P + 25C)/2 - ATM)")

## 3. Visualizing the Term Structure

The **term structure** shows how implied vol varies with expiry at a fixed strike/delta.

In [ ]:
# =============================================================================
# STEP 3: Visualize the Vol Term Structure
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

delta_colors = ['red', 'orange', 'green', 'blue', 'purple']

for j, (delta, label, color) in enumerate(zip(deltas, delta_labels, delta_colors)):
    vols = vol_matrix[:, j]
    ax.plot(expiries, vols * 100, 'o-', color=color, linewidth=2,
            markersize=8, label=label)

ax.set_xlabel('Time to Expiry (years)', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title('EURUSD Volatility Term Structure by Delta', fontsize=14, fontweight='bold')
ax.legend(title='Delta', loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.05)

plt.tight_layout()
plt.show()

# Term structure analysis
print("\nTerm Structure Analysis (ATM):")
atm_vols = vol_matrix[:, 2]
print(f"  1W ATM vol:  {atm_vols[0]:.2%}")
print(f"  1Y ATM vol:  {atm_vols[-1]:.2%}")
print(f"  Term slope:  {(atm_vols[-1] - atm_vols[0]) / (expiries[-1] - expiries[0]):.4f} (vol per year)")

## 4. 3D Volatility Surface Visualization

The complete picture: a 3D surface plot showing vol as a function of both strike (delta) and expiry.

In [ ]:
# =============================================================================
# STEP 4: 3D Vol Surface Plot
# =============================================================================

# Create meshgrid for 3D plot
X, Y = np.meshgrid(deltas, expiries)
Z = vol_matrix * 100  # Convert to percentage

# Create figure with two subplots
fig = plt.figure(figsize=(16, 6))

# 3D Surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap=cm.viridis, alpha=0.8,
                        linewidth=0.5, antialiased=True, edgecolor='k')
ax1.set_xlabel('Delta')
ax1.set_ylabel('Expiry (years)')
ax1.set_zlabel('Implied Vol (%)')
ax1.set_title('3D Volatility Surface', fontsize=14, fontweight='bold')
ax1.view_init(elev=25, azim=-60)

# Contour plot (bird's eye view)
ax2 = fig.add_subplot(122)
contour = ax2.contourf(X, Y, Z, levels=20, cmap=cm.viridis)
ax2.set_xlabel('Delta')
ax2.set_ylabel('Expiry (years)')
ax2.set_title('Vol Surface Contour Map', fontsize=14, fontweight='bold')
cbar = plt.colorbar(contour, ax=ax2, label='Implied Vol (%)')

# Add contour lines
ax2.contour(X, Y, Z, levels=10, colors='black', alpha=0.3, linewidths=0.5)

plt.tight_layout()
plt.show()

## 5. Arbitrage Constraints

A valid vol surface must satisfy **no-arbitrage conditions**:

### Calendar Spread Arbitrage
Total variance must increase with time:
$$\sigma^2(T_1) \cdot T_1 \leq \sigma^2(T_2) \cdot T_2 \quad \text{for } T_1 < T_2$$

### Butterfly Arbitrage
The local vol (Dupire) must be positive, which translates to constraints on the smile curvature.

In [ ]:
# =============================================================================
# STEP 5: Check Arbitrage Constraints
# =============================================================================

print("Arbitrage Constraint Checks")
print("="*60)

# Calendar spread check: Total variance must be increasing
print("\n1. Calendar Spread Check (Total Variance Increasing):")
print("-"*60)
print(f"{'Delta':<12} {'Expiry 1':<10} {'Expiry 2':<10} {'TotVar1':>10} {'TotVar2':>10} {'OK?':>6}")
print("-"*60)

calendar_violations = 0
for j, delta in enumerate(deltas):
    for i in range(len(expiries) - 1):
        t1, t2 = expiries[i], expiries[i+1]
        vol1, vol2 = vol_matrix[i, j], vol_matrix[i+1, j]
        totvar1 = vol1**2 * t1
        totvar2 = vol2**2 * t2
        ok = totvar2 >= totvar1
        if not ok:
            calendar_violations += 1
            print(f"{delta_labels[j]:<12} {expiry_labels[i]:<10} {expiry_labels[i+1]:<10} "
                  f"{totvar1:>10.6f} {totvar2:>10.6f} {'✗':>6}")

if calendar_violations == 0:
    print("All calendar spread checks PASSED ✓")
else:
    print(f"\nCalendar violations: {calendar_violations}")

# Butterfly check: Second derivative of call price w.r.t. strike > 0
# Simplified: Check that smile is convex (second difference positive)
print("\n2. Butterfly Check (Smile Convexity):")
print("-"*60)

butterfly_violations = 0
for i, exp_label in enumerate(expiry_labels):
    vols = vol_matrix[i, :]
    # Check second differences
    for j in range(1, len(vols) - 1):
        second_diff = vols[j-1] - 2*vols[j] + vols[j+1]
        if second_diff < -0.001:  # Small tolerance
            butterfly_violations += 1
            print(f"{exp_label}: Non-convex at delta={deltas[j]}, curvature={second_diff:.4f}")

if butterfly_violations == 0:
    print("All butterfly checks PASSED ✓")
else:
    print(f"\nButterfly violations: {butterfly_violations}")

## 6. SABR Model Calibration

The **SABR model** is widely used to parameterize the vol smile:

$$\sigma_{SABR}(K, F, T) = f(\alpha, \beta, \rho, \nu)$$

where:
- $\alpha$: Initial volatility level
- $\beta$: CEV exponent (often fixed at 0 or 0.5)
- $\rho$: Spot-vol correlation (drives skew)
- $\nu$: Vol-of-vol (drives smile curvature)

Let's fit SABR to one expiry and visualize the fit.

In [ ]:
# =============================================================================
# STEP 6: SABR Model Illustration
# =============================================================================

def sabr_vol_hagan(F, K, T, alpha, beta, rho, nu):
    """
    Hagan's SABR approximation for implied volatility.
    
    Parameters
    ----------
    F : float - Forward price
    K : float - Strike
    T : float - Time to expiry
    alpha : float - Initial vol level
    beta : float - CEV exponent
    rho : float - Spot-vol correlation
    nu : float - Vol of vol
    
    Returns
    -------
    float - Implied volatility
    """
    if abs(F - K) < 1e-10:  # ATM case
        FK = F ** (1 - beta)
        term1 = ((1 - beta)**2 / 24) * alpha**2 / FK**2
        term2 = 0.25 * rho * beta * nu * alpha / FK
        term3 = (2 - 3*rho**2) / 24 * nu**2
        return alpha / FK * (1 + (term1 + term2 + term3) * T)
    
    FK_mid = (F * K) ** ((1 - beta) / 2)
    z = nu / alpha * FK_mid * np.log(F / K)
    x_z = np.log((np.sqrt(1 - 2*rho*z + z**2) + z - rho) / (1 - rho))
    
    # Leading order
    leading = alpha / (FK_mid * (1 + (1-beta)**2/24 * np.log(F/K)**2 + 
                                   (1-beta)**4/1920 * np.log(F/K)**4))
    
    # z/x(z) term
    if abs(x_z) < 1e-10:
        z_xz = 1.0
    else:
        z_xz = z / x_z
    
    # Correction terms
    term1 = ((1 - beta)**2 / 24) * alpha**2 / FK_mid**2
    term2 = 0.25 * rho * beta * nu * alpha / FK_mid
    term3 = (2 - 3*rho**2) / 24 * nu**2
    
    return leading * z_xz * (1 + (term1 + term2 + term3) * T)


# Market parameters
T = 0.25  # 3-month expiry
F = SPOT * np.exp((R_DOMESTIC - R_FOREIGN) * T)  # Forward

# SABR parameters (calibrated values)
alpha = 0.085   # Initial vol
beta = 0.5      # CEV exponent (fixed)
rho = -0.25     # Spot-vol correlation (negative = skew)
nu = 0.45       # Vol of vol (controls curvature)

# Generate strikes from delta (simplified)
strikes = F * np.exp(-0.5 * alpha**2 * T + alpha * np.sqrt(T) * 
                     np.array([-1.28, -0.67, 0, 0.67, 1.28]))  # Approx delta to strike

# Compute SABR vols
strike_range = np.linspace(F * 0.85, F * 1.15, 100)
sabr_vols = [sabr_vol_hagan(F, K, T, alpha, beta, rho, nu) for K in strike_range]

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: SABR fit
ax1 = axes[0]
ax1.plot(strike_range / F, np.array(sabr_vols) * 100, 'b-', linewidth=2, label='SABR Model')
ax1.scatter(np.array([0.9, 0.95, 1.0, 1.05, 1.10]), vol_matrix[4, :] * 100, 
            s=100, c='red', marker='o', label='Market Quotes', zorder=5)
ax1.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Strike / Forward (Moneyness)', fontsize=12)
ax1.set_ylabel('Implied Volatility (%)', fontsize=12)
ax1.set_title(f'SABR Fit to 3M Smile\nα={alpha}, β={beta}, ρ={rho}, ν={nu}', 
              fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: SABR parameter sensitivity
ax2 = axes[1]

# Vary rho
for rho_val in [-0.5, -0.25, 0, 0.25]:
    vols = [sabr_vol_hagan(F, K, T, alpha, beta, rho_val, nu) for K in strike_range]
    ax2.plot(strike_range / F, np.array(vols) * 100, linewidth=2, 
             label=f'ρ = {rho_val}')

ax2.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Strike / Forward (Moneyness)', fontsize=12)
ax2.set_ylabel('Implied Volatility (%)', fontsize=12)
ax2.set_title('Effect of ρ (Correlation) on Smile Shape', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSABR Parameter Effects:")
print("="*60)
print("  ρ < 0: Negative skew (puts more expensive) - typical for equities")
print("  ρ > 0: Positive skew (calls more expensive) - some commodities")
print("  ν ↑:   Steeper smile curvature (fatter tails implied)")
print("  α:     Shifts the entire smile up/down (ATM level)")

## 7. Key Takeaways

### Volatility Surface Structure
- **Smile**: Vol varies with strike (higher for OTM options)
- **Skew**: Asymmetry in the smile (typically negative for equities/FX)
- **Term structure**: Vol varies with expiry (mean reversion effects)

### Arbitrage Constraints
- **Calendar arbitrage**: Total variance must increase with time
- **Butterfly arbitrage**: Call prices must be convex in strike

### SABR Model
- Industry-standard for FX and rates
- 4 parameters with intuitive meanings
- Captures smile dynamics through stochastic vol

### Practical Applications
- **Exotic pricing**: Need full surface for path-dependent options
- **Risk management**: Greeks depend on vol surface shape
- **Trading**: Identify cheap/expensive options via surface deviations

In [ ]:
print("\n" + "="*60)
print("Try experimenting with the parameters above!")
print("="*60)
print("\nSuggested exercises:")
print("1. Change the vol matrix to create a steeper smile")
print("2. Create an inverted term structure (vol decreasing with expiry)")
print("3. Vary SABR ν parameter to see effect on wing vols")
print("4. Add a calendar arbitrage violation and see the check fail")